1. Dosya İnceleme

In [1]:
import os
import pandas as pd

data_dir = r"C:\Users\stasd\Desktop\zero-day-detection\data"

# Klasördeki tüm CSV'leri listele
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
for f in csv_files:
    print(f)

02-14-2018.csv
02-15-2018.csv
02-16-2018.csv
02-20-2018.csv
02-21-2018.csv
02-22-2018.csv
02-23-2018.csv
02-28-2018.csv
03-01-2018.csv
03-02-2018.csv


2. Her Dosyanın İçine Bak (Birleştirmeden Önce)

In [2]:
for f in csv_files:
    path = os.path.join(data_dir, f)
    temp = pd.read_csv(path, encoding='latin-1', nrows=5)
    print(f"\n{'='*40}")
    print(f"Dosya: {f}")
    print(f"Sütunlar: {temp.columns.tolist()}")


Dosya: 02-14-2018.csv
Sütunlar: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b 

3. Birleştirme

In [1]:
import pandas as pd
import glob
import gc
import os

# 1. Temizlenmiş Parquet dosyaları için yeni bir klasör oluşturuyoruz
output_dir = '../data/cleaned_parquets'
os.makedirs(output_dir, exist_ok=True)

# Windows yolları için glob ile tüm csv'leri al
csv_files = glob.glob('../data/*.csv')
cols_to_drop = ['Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Timestamp']

print("Dosyalar TEK TEK işlenip diske kaydediliyor. RAM tamamen güvende!")

for i, file in enumerate(csv_files):
    # Dosya adını al (Windows ve Mac/Linux uyumlu)
    file_name = os.path.basename(file)
    print(f"\n[{i+1}/{len(csv_files)}] İşleniyor: {file_name}")
    
    # 2. Sadece o anki CSV'yi parçalar halinde oku
    chunk_iter = pd.read_csv(file, chunksize=250000, low_memory=False) 
    temp_chunks = []
    
    for chunk in chunk_iter:
        chunk.columns = chunk.columns.str.strip()
        existing_cols = [c for c in cols_to_drop if c in chunk.columns]
        chunk.drop(columns=existing_cols, inplace=True)
        
        for col in chunk.columns:
            if col != 'Label':
                chunk[col] = pd.to_numeric(chunk[col], errors='coerce')
        
        numeric_cols = chunk.select_dtypes(include=['float64', 'int64']).columns
        chunk[numeric_cols] = chunk[numeric_cols].astype('float32')
        
        temp_chunks.append(chunk)
        
    # 3. Sadece BİR CSV dosyasının parçalarını birleştir
    single_csv_df = pd.concat(temp_chunks, ignore_index=True)
    
    # 4. Bu tekil dosyayı Parquet olarak kaydet
    save_path = os.path.join(output_dir, f"part_{i}.parquet")
    single_csv_df.to_parquet(save_path)
    print(f"--> Başarılı! {save_path} olarak SSD'ye yazıldı.")
    
    # 5. En kritik adım: Değişkenleri sil ve RAM'i ZORLA boşalt
    del temp_chunks
    del single_csv_df
    gc.collect()

print("\nHarika! Tüm dosyalar temizlendi ve RAM hiç zorlanmadan ayrı ayrı kaydedildi.")

Dosyalar TEK TEK işlenip diske kaydediliyor. RAM tamamen güvende!

[1/10] İşleniyor: 02-14-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_0.parquet olarak SSD'ye yazıldı.

[2/10] İşleniyor: 02-15-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_1.parquet olarak SSD'ye yazıldı.

[3/10] İşleniyor: 02-16-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_2.parquet olarak SSD'ye yazıldı.

[4/10] İşleniyor: 02-20-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_3.parquet olarak SSD'ye yazıldı.

[5/10] İşleniyor: 02-21-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_4.parquet olarak SSD'ye yazıldı.

[6/10] İşleniyor: 02-22-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_5.parquet olarak SSD'ye yazıldı.

[7/10] İşleniyor: 02-23-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_6.parquet olarak SSD'ye yazıldı.

[8/10] İşleniyor: 02-28-2018.csv
--> Başarılı! ../data/cleaned_parquets\part_7.parquet olarak SSD'ye yazıldı.

[9/10] İşleniyor: 03-01-2018.csv
--> Başarılı